# World Cup Model: Hypothesis Tests

Each idea is stated as a question with an explicit null hypothesis, a test at alpha = 0.05, and a verdict.

Outline: imports, configuration, data, then one hypothesis per section, ending in a feature scorecard.

## Imports

In [1]:
import sys, pathlib
_root = pathlib.Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.calibration import CalibratedClassifierCV

from src.worldcup import config as C
from src.worldcup.features import build_dataset, edition_table_filled, FEATURES
from src.worldcup.model import train_eval, symmetrize
from src.worldcup.squads import confirmed_strength_table
from src.worldcup.cross_tournament import analyse_correlation
from src.worldcup.defense_study import analyse as defense_analyse, attach_ratings, tournament_outcomes
from src.analysis.evaluation import log_loss as ll


## Configuration

In [2]:
RANDOM_STATE = 7
ALPHA = 0.05
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)


## Data

In [3]:
bundle_2022 = build_dataset(2022)
train_2022, test_2022 = bundle_2022["train"], bundle_2022["test"]
(train_2022.shape, test_2022.shape)


((1396, 14), (64, 16))

## Hypotheses tested

### Does adding EA FC squad strength improve on an Elo-only model?

Null: per-match log-loss of the full model equals the Elo-only model; alternative: the full model is lower. alpha = 0.05.

In [4]:
def fit_pipe(tr, feats):
    sym = symmetrize(tr, feats)
    m = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    m.fit(sym[feats], sym["y"])
    return m

def per_match_ll(model, te, feats):
    p = model.predict_proba(te[feats])
    y = te["y"].to_numpy()
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))


In [5]:
ELO = ["d_elo", "home_field"]
full_ll, elo_ll = [], []
table = []
for yr in (2018, 2022):
    d = build_dataset(yr)
    tr, te = d["train"], d["test"]
    pf = per_match_ll(fit_pipe(tr, list(FEATURES)), te, list(FEATURES))
    pe = per_match_ll(fit_pipe(tr, ELO), te, ELO)
    full_ll.append(pf); elo_ll.append(pe)
    table.append({"world_cup": yr, "full_logloss": pf.mean(), "elo_only_logloss": pe.mean()})
full_ll = np.concatenate(full_ll); elo_ll = np.concatenate(elo_ll)
stat, p = stats.wilcoxon(full_ll, elo_ll, alternative="less")
print(pd.DataFrame(table).round(4).to_string(index=False))
print("Wilcoxon signed-rank (full < Elo-only): statistic=%.1f p=%.4f -> %s"
      % (stat, p, "reject null" if p < ALPHA else "fail to reject"))


 world_cup  full_logloss  elo_only_logloss
      2018        0.9500            0.9866
      2022        1.0228            1.0516
Wilcoxon signed-rank (full < Elo-only): statistic=3198.0 p=0.0135 -> reject null


Verdict: yes

### Does recent form add anything once Elo is already in the model?

Null: recent competitive form is uncorrelated with World Cup points-per-game. alpha = 0.05.

In [6]:
corr = analyse_correlation().set_index("predictor")
recent_r = corr.loc["recent competitive ppg (24m)", "pearson"]
elo_r = corr.loc["pre-tournament Elo (benchmark)", "pearson"]
n_recent = int(corr.loc["recent competitive ppg (24m)", "n"])
t = recent_r * np.sqrt((n_recent - 2) / (1 - recent_r ** 2))
p = 2 * stats.t.sf(abs(t), n_recent - 2)
print("recent form vs WC ppg: pearson %+.3f (n=%d), p=%.3f -> %s"
      % (recent_r, n_recent, p, "reject null" if p < ALPHA else "fail to reject"))
print("Elo vs WC ppg: pearson %+.3f (benchmark)" % elo_r)


recent form vs WC ppg: pearson +0.091 (n=221), p=0.178 -> fail to reject
Elo vs WC ppg: pearson +0.516 (benchmark)


Verdict: no

### Does flagging African (CAF) teams help or hurt the model?

Comparison across the only two rated World Cups; too few tournaments for a formal test, so the sign and size of the change are read directly.

In [7]:
CAF = {"Senegal", "Morocco", "Tunisia", "Cameroon", "Ghana", "Nigeria", "Egypt",
       "Algeria", "Ivory Coast", "Mali", "South Africa", "Cape Verde",
       "Burkina Faso", "Congo DR", "DR Congo"}
def caf_dummy(df):
    return df["team1"].isin(CAF).astype(int) - df["team2"].isin(CAF).astype(int)

def test_logloss(tr, te, feats):
    m = fit_pipe(tr, feats)
    return ll(m.predict_proba(te[feats]), te["y"].to_numpy())


In [8]:
h3 = []
for yr in (2018, 2022):
    d = build_dataset(yr)
    tr, te = d["train"].copy(), d["test"].copy()
    tr["d_caf"], te["d_caf"] = caf_dummy(tr), caf_dummy(te)
    base = test_logloss(tr, te, list(FEATURES))
    with_caf = test_logloss(tr, te, list(FEATURES) + ["d_caf"])
    h3.append({"world_cup": yr, "log_loss_base": base,
               "log_loss_with_caf": with_caf, "delta": with_caf - base})
h3 = pd.DataFrame(h3).round(4)
h3


,world_cup,log_loss_base,log_loss_with_caf,delta
0,2018,0.9500,0.9649,0.0149
1,2022,1.0228,1.0158,-0.0071


Verdict: no consistent gain

### Do confirmed 26-man squads differ significantly from the nationality-pool proxy?

Null: the median difference (confirmed minus proxy) across nations is zero. alpha = 0.05.

In [9]:
conf, rep = confirmed_strength_table()
proxy = edition_table_filled(2026)
delta = (conf["ovr_top23"] - proxy["ovr_top23"]).dropna()
stat, p = stats.wilcoxon(delta)
print("nations compared: %d, mean delta %.2f, median delta %.2f"
      % (len(delta), delta.mean(), delta.median()))
print("Wilcoxon signed-rank: statistic=%.1f p=%.4f -> %s"
      % (stat, p, "reject null" if p < ALPHA else "fail to reject"))
focus = pd.DataFrame({"proxy": proxy["ovr_top23"], "confirmed": conf["ovr_top23"]})
focus.loc[["Spain", "France", "Brazil", "Argentina", "England"]].round(1)


nations compared: 46, mean delta -1.31, median delta -1.20
Wilcoxon signed-rank: statistic=39.5 p=0.0000 -> reject null


,proxy,confirmed
Spain,85.0,84.0
France,85.2,84.0
Brazil,83.9,78.7
Argentina,82.7,82.0
England,84.0,82.3


Verdict: yes

### Does sigmoid (Platt) or isotonic calibration give more robust probabilities?

No formal test; the failure mode is shown directly by comparing log-loss and the smallest draw probability each method assigns.

In [10]:
def fit_cal(method, tr, te, feats):
    tr = tr.sort_values("date").reset_index(drop=True)
    cut = int(len(tr) * 0.85)
    base = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    fit_sym = symmetrize(tr.iloc[:cut], feats)
    base.fit(fit_sym[feats], fit_sym["y"])
    cal = CalibratedClassifierCV(base, method=method, cv="prefit")
    cal_sym = symmetrize(tr.iloc[cut:], feats)
    cal.fit(cal_sym[feats], cal_sym["y"])
    p = cal.predict_proba(te[feats])
    return ll(p, te["y"].to_numpy()), float(p[:, 1].min())


In [11]:
h5 = []
for yr in (2018, 2022):
    d = build_dataset(yr)
    te = d["test"]
    s_ll, s_min = fit_cal("sigmoid", d["train"], te, list(FEATURES))
    i_ll, i_min = fit_cal("isotonic", d["train"], te, list(FEATURES))
    h5.append({"world_cup": yr, "sigmoid_logloss": s_ll, "isotonic_logloss": i_ll,
               "isotonic_min_draw_p": i_min, "sigmoid_min_draw_p": s_min})
h5 = pd.DataFrame(h5).round(4)
h5


,world_cup,sigmoid_logloss,isotonic_logloss,isotonic_min_draw_p,sigmoid_min_draw_p
0,2018,0.9545,2.1354,0.0,0.2464
1,2022,1.0109,0.9917,0.0,0.2153


Verdict: sigmoid is the safer choice

### Does a strong defence (goalkeeper plus defenders) win tournaments beyond overall quality?

Null: defensive tilt (defence minus overall) is uncorrelated with tournament points-per-game. alpha = 0.05.

In [12]:
study = defense_analyse(attach_ratings(tournament_outcomes()))
r_overall, p_overall = study["corr_overall_ppg"]
r_tilt, p_tilt = study["corr_deftilt_ppg"]
print("rows %d, tournaments %d, champions %d"
      % (study["n_rows"], study["n_instances"], study["n_champions"]))
print("overall      vs PPG: r=%+.3f p=%.3f" % (r_overall, p_overall))
print("defence TILT vs PPG: r=%+.3f p=%.3f -> %s"
      % (r_tilt, p_tilt, "reject null" if p_tilt < ALPHA else "fail to reject"))
print("champions defence tilt %.2f vs field %.2f"
      % (study["champ_mean_def_tilt"], study["field_mean_def_tilt"]))


rows 289, tournaments 19, champions 15
overall      vs PPG: r=+0.576 p=0.000
defence TILT vs PPG: r=-0.104 p=0.077 -> fail to reject
champions defence tilt -0.06 vs field 0.79


Verdict: no

### Does continental-tournament form (Euro, Copa) predict World Cup performance?

Null: continental campaign points-per-game is uncorrelated with World Cup points-per-game. alpha = 0.05.

In [13]:
corr = analyse_correlation().set_index("predictor")
r_cont = corr.loc["continental campaign ppg", "pearson"]
n_cont = int(corr.loc["continental campaign ppg", "n"])
t = r_cont * np.sqrt((n_cont - 2) / (1 - r_cont ** 2))
p = 2 * stats.t.sf(abs(t), n_cont - 2)
print("continental form vs WC ppg: pearson %+.3f (n=%d), p=%.3f -> %s"
      % (r_cont, n_cont, p, "reject null" if p < ALPHA else "fail to reject"))
analyse_correlation().round(3)


continental form vs WC ppg: pearson -0.020 (n=195), p=0.777 -> fail to reject


,predictor,n,pearson,spearman
0,continental campaign ppg,195,-0.020,-0.047
1,recent competitive ppg (24m),221,0.091,0.086
2,pre-tournament Elo (benchmark),224,0.516,0.534


Verdict: no

### Should host advantage apply to non-host teams at a neutral World Cup?

Design check rather than a statistical test: confirm the home edge is carried only by the host inside the tournament.

In [14]:
wc_hf = test_2022[["team1", "team2", "home_field"]]
host_rows = wc_hf[wc_hf["home_field"] != 0]
host_teams = sorted(set(host_rows["team1"]) | set(host_rows["team2"]))
print("World Cup 2022 test matches with a home edge: %d of %d" % (len(host_rows), len(wc_hf)))
print("teams appearing in those matches:", host_teams)


World Cup 2022 test matches with a home edge: 3 of 64
teams appearing in those matches: ['Ecuador', 'Netherlands', 'Qatar', 'Senegal']


Verdict: hosts only

## Feature scorecard

In [15]:
scorecard = pd.DataFrame([
    ("EA FC squad strength", "Beats Elo-only?", "WORKED",
     "Wilcoxon p<0.05 on per-match log-loss"),
    ("Recent form (24m ppg)", "Adds beyond Elo?", "NO",
     "pearson %+.2f vs Elo %+.2f" % (recent_r, elo_r)),
    ("Confederation (CAF) flag", "Helps African teams?", "NO (noise)",
     "log-loss change %+.3f (2018), %+.3f (2022)" % (
         h3.set_index("world_cup").loc[2018, "delta"],
         h3.set_index("world_cup").loc[2022, "delta"])),
    ("Confirmed 26-man squads", "Differ from proxy?", "MATTERS",
     "Wilcoxon p<0.05, deep nations fall"),
    ("Sigmoid calibration", "Better than isotonic?", "SIGMOID",
     "isotonic %.2f log-loss in 2018 (draw p -> 0)" %
     h5.set_index("world_cup").loc[2018, "isotonic_logloss"]),
    ("Defensive tilt", "Wins titles beyond quality?", "NO",
     "tilt r=%+.2f p=%.2f" % study["corr_deftilt_ppg"]),
    ("Continental form", "Predicts the World Cup?", "NO",
     "pearson near zero"),
    ("Host advantage", "Apply to non-hosts?", "HOSTS ONLY",
     "home edge only for host nations"),
], columns=["feature", "question", "verdict", "evidence"])
scorecard.to_csv(C.OUT / "feature_scorecard.csv", index=False)
scorecard


,feature,question,verdict,evidence
0,EA FC squad strength,Beats Elo-only?,WORKED,Wilcoxon p<0.05 on per-match log-loss
1,Recent form (24m ppg),Adds beyond Elo?,NO,pearson +0.09 vs Elo +0.52
2,Confederation (CAF) flag,Helps African teams?,NO (noise),"log-loss change +0.015 (2018), -0.007 (2022)"
3,Confirmed 26-man squads,Differ from proxy?,MATTERS,"Wilcoxon p<0.05, deep nations fall"
4,Sigmoid calibration,Better than isotonic?,SIGMOID,isotonic 2.14 log-loss in 2018 (draw p -> 0)
5,Defensive tilt,Wins titles beyond quality?,NO,tilt r=-0.10 p=0.08
6,Continental form,Predicts the World Cup?,NO,pearson near zero
7,Host advantage,Apply to non-hosts?,HOSTS ONLY,home edge only for host nations


## Conclusion

Squad strength plus Elo is the engine; form, a confederation flag, defensive tilt, and continental form add no usable signal, and probabilities are calibrated with sigmoid scaling.